# 01 - EDA & Data Preparation

Eksplorasi dataset: distribusi kelas, statistik resolusi, visualisasi seluruh komoditas,
distribusi warna HSV fresh vs rotten, dan visualisasi pipeline preprocessing
(SSR, enhancement, segmentasi). Diakhiri dengan pembuatan split stratified 70/15/15.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.enhancement import apply_enhancement
from src.preprocessing import check_integrity, preprocess_from_array
from src.segmentation import segment_fruit
from src.utils import (
    build_dataset_index,
    get_project_paths,
    make_splits,
    save_splits,
    set_seed,
)

set_seed(42)
paths = get_project_paths()
print("Sumber dataset:", RAW_DATA_DIR)


## 1. Statistik Dataset

In [ ]:
df = build_dataset_index(RAW_DATA_DIR)
print(f"Total citra  : {len(df)}")
print(f"Komoditas    : {df['commodity'].nunique()}")
print(f"Label        : {df['label'].value_counts().to_dict()}")
print(f"Imbalance ratio: {df['label'].value_counts().max() / df['label'].value_counts().min():.2f}x")
df.head()


In [ ]:
# Distribusi jumlah citra per komoditas x label
counts = df.groupby(["commodity", "label"]).size().unstack(fill_value=0)
counts["total"] = counts.sum(axis=1)
counts = counts.sort_values("total", ascending=False)
display(counts)
print(f"\nRata-rata per komoditas: {counts['total'].mean():.0f} citra")
print(f"Min: {counts['total'].min()} | Max: {counts['total'].max()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["label"].value_counts().plot(
    kind="bar", ax=axes[0], color=["green", "brown"], rot=0,
    title="Distribusi Global Kelas"
)
axes[0].set_ylabel("Jumlah citra")
df.groupby(["commodity", "label"]).size().unstack(fill_value=0).sort_index().plot(
    kind="bar", stacked=True, ax=axes[1],
    color=["green", "brown"],
    title="Distribusi per Komoditas"
)
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Check image resolution distribution (sample 300)
sample_paths = df["filepath"].sample(min(300, len(df)), random_state=42).tolist()
shapes = []
for fp in sample_paths:
    img = cv2.imread(fp)
    if img is not None:
        shapes.append(img.shape[:2])
shape_df = pd.DataFrame(shapes, columns=["height", "width"])
print("Statistik resolusi citra (sample 300):")
print(shape_df.describe().round(1))
n_unique = len(shape_df.drop_duplicates())
print(f"\nJumlah ukuran unik: {n_unique}")
if n_unique <= 10:
    print("Ukuran unik:", shape_df.drop_duplicates().values.tolist())


## 2. Sampel Visual Semua Komoditas

In [ ]:
commodities = sorted(df['commodity'].unique())
ncols, nrows = 2, len(commodities)
fig, axes = plt.subplots(nrows, ncols, figsize=(6, nrows * 2.2))
for i, comm in enumerate(commodities):
    for j, label in enumerate(["fresh", "rotten"]):
        ax = axes[i, j]
        subset = df[(df["commodity"] == comm) & (df["label"] == label)]
        if not subset.empty:
            img = cv2.imread(subset.iloc[0]["filepath"])
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{comm}\n({label})", fontsize=7)
        ax.axis("off")
plt.suptitle("Sampel per Komoditas — kiri: fresh, kanan: rotten", fontsize=10)
plt.tight_layout()
plt.show()


## 3. Pemeriksaan Integritas

In [ ]:
corrupt = [fp for fp in df["filepath"] if not check_integrity(fp)]
print(f"Citra corrupt / tidak terbaca: {len(corrupt)}")
if corrupt:
    for fp in corrupt[:5]:
        print(' ', fp)


## 4. Distribusi Warna HSV: Fresh vs Rotten

Histogram HSV rata-rata dari 500 sampel tiap kelas. Perbedaan distribusi
Hue (H), Saturation (S), dan Value (V) antara fresh dan rotten
**memotivasi penggunaan HSV histogram sebagai fitur utama** dalam pipeline klasifikasi.

In [ ]:
def _hsv_histograms(filepaths, n_sample=500):
    """Compute mean HSV histograms over a sample of images."""
    h_acc = np.zeros(180, dtype=np.float64)
    s_acc = np.zeros(256, dtype=np.float64)
    v_acc = np.zeros(256, dtype=np.float64)
    count = 0
    for fp in filepaths[:n_sample]:
        img = cv2.imread(fp)
        if img is None:
            continue
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h_acc += cv2.calcHist([hsv], [0], None, [180], [0, 180]).flatten()
        s_acc += cv2.calcHist([hsv], [1], None, [256], [0, 256]).flatten()
        v_acc += cv2.calcHist([hsv], [2], None, [256], [0, 256]).flatten()
        count += 1
    if count:
        h_acc /= count; s_acc /= count; v_acc /= count
    return h_acc, s_acc, v_acc

fresh_fps  = df[df["label"] == "fresh"]["filepath"].tolist()
rotten_fps = df[df["label"] == "rotten"]["filepath"].tolist()
h_f, s_f, v_f = _hsv_histograms(fresh_fps)
h_r, s_r, v_r = _hsv_histograms(rotten_fps)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_data = [
    (h_f, h_r, "Hue (H)", "Nilai Hue (0–179)"),
    (s_f, s_r, "Saturation (S)", "Nilai Saturasi (0–255)"),
    (v_f, v_r, "Value / Kecerahan (V)", "Nilai Value (0–255)"),
]
for ax, (fresh_hist, rotten_hist, title, xlabel) in zip(axes, channel_data):
    ax.plot(fresh_hist,  color="green", label="Fresh",  alpha=0.8, linewidth=1.5)
    ax.plot(rotten_hist, color="brown", label="Rotten", alpha=0.8, linewidth=1.5)
    ax.set_title(f"Distribusi {title}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Rata-rata frekuensi piksel")
    ax.legend()

plt.suptitle(
    "Distribusi Warna HSV: Fresh vs Rotten\n"
    "Perbedaan pada H, S, V memotivasi HSV histogram sebagai fitur warna utama",
    fontsize=10,
)
plt.tight_layout()
plt.show()


## 5. Visualisasi Pipeline Preprocessing

Menampilkan efek tiap tahap preprocessing: SSR, enhancement (CLAHE/gamma),
dan segmentasi Otsu. Membantu memahami transformasi citra sebelum ekstraksi fitur.

In [ ]:
# Pick one representative fresh and one rotten sample for all visualizations below
fp_fresh  = df[df["label"] == "fresh"].iloc[0]["filepath"]
fp_rotten = df[df["label"] == "rotten"].iloc[0]["filepath"]
print("Fresh sample :", fp_fresh)
print("Rotten sample:", fp_rotten)


### 5a. Efek Single-Scale Retinex (SSR)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, (fp, label) in enumerate([(fp_fresh, "Fresh"), (fp_rotten, "Rotten")]):
    img_bgr  = cv2.imread(fp)
    original = cv2.resize(img_bgr, (224, 224))
    restored = preprocess_from_array(img_bgr, apply_restoration=True)
    diff     = cv2.absdiff(original, restored)
    diff_vis = np.clip(diff.astype(np.float32) * 5, 0, 255).astype(np.uint8)
    for col, (image, title, cmap) in enumerate([
        (cv2.cvtColor(original,  cv2.COLOR_BGR2RGB), "Original (resize 224×224)", None),
        (cv2.cvtColor(restored,  cv2.COLOR_BGR2RGB), "Setelah SSR", None),
        (diff_vis,                                   "Difference ×5 (panas=berubah)", "hot"),
    ]):
        axes[row, col].imshow(image, cmap=cmap)
        axes[row, col].set_title(f"{label} – {title}", fontsize=9)
        axes[row, col].axis("off")
plt.suptitle(
    "Efek Single-Scale Retinex (SSR)\n"
    "Koreksi pencahayaan non-uniform: area gelap dinaikkan, area sangat terang direduksi",
    fontsize=10,
)
plt.tight_layout()
plt.show()


### 5b. Perbandingan Enhancement (none vs CLAHE vs Gamma)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (fp, label) in enumerate([(fp_fresh, "Fresh"), (fp_rotten, "Rotten")]):
    img_bgr = cv2.imread(fp)
    ssr_img = preprocess_from_array(img_bgr, apply_restoration=True)
    columns = [
        (cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB), "Original"),
        (cv2.cvtColor(ssr_img,                          cv2.COLOR_BGR2RGB), "SSR (no enhance)"),
        (cv2.cvtColor(apply_enhancement(ssr_img, "clahe"), cv2.COLOR_BGR2RGB), "SSR + CLAHE"),
        (cv2.cvtColor(apply_enhancement(ssr_img, "gamma"), cv2.COLOR_BGR2RGB), "SSR + Gamma"),
    ]
    for col, (image, title) in enumerate(columns):
        axes[row, col].imshow(image)
        axes[row, col].set_title(f"{label}\n{title}", fontsize=9)
        axes[row, col].axis("off")
plt.suptitle(
    "Perbandingan Enhancement Method\n"
    "E* (enhancement terbaik) dipilih otomatis berdasarkan validation F1 pada S2–S4",
    fontsize=10,
)
plt.tight_layout()
plt.show()


### 5c. Segmentasi Otsu + Morfologi

In [ ]:
# Use E*='clahe' as a stand-in for visualization (actual E* resolved at runtime in nb02)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (fp, label) in enumerate([(fp_fresh, "Fresh"), (fp_rotten, "Rotten")]):
    img_bgr  = cv2.imread(fp)
    ssr_img  = preprocess_from_array(img_bgr, apply_restoration=True)
    enhanced = apply_enhancement(ssr_img, "clahe")
    masked, binary_mask, obj_ratio, used_fallback = segment_fruit(enhanced)
    fallback_note = " [fallback]" if used_fallback else f" ({obj_ratio:.0%} fg)"
    columns = [
        (cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB), "Original", None),
        (cv2.cvtColor(ssr_img,  cv2.COLOR_BGR2RGB), "SSR", None),
        (binary_mask, f"Mask Otsu{fallback_note}", "gray"),
        (cv2.cvtColor(masked, cv2.COLOR_BGR2RGB), "Hasil Segmentasi", None),
    ]
    for col, (image, title, cmap) in enumerate(columns):
        axes[row, col].imshow(image, cmap=cmap)
        axes[row, col].set_title(f"{label}\n{title}", fontsize=9)
        axes[row, col].axis("off")
plt.suptitle(
    "Segmentasi Otsu + Morfologi (ellipse kernel open→close + largest contour)\n"
    "Memisahkan ROI buah/sayur dari background; mereduksi noise piksel non-objek",
    fontsize=10,
)
plt.tight_layout()
plt.show()


## 6. Pembuatan Split Stratified (70/15/15)

In [ ]:
train, val, test = make_splits(df)
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Stratifikasi train: {train['label'].value_counts().to_dict()}")
print(f"Stratifikasi test : {test['label'].value_counts().to_dict()}")
split_path = save_splits(train, val, test, paths["splits"])
print(f"Split disimpan ke: {split_path}")
